# Diabetes Prediction Model — XGBoost
Dataset: Pima Indians Diabetes (OpenML)
Task: Binary classification

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import xgboost as xgb
import shap
import joblib
import os
MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)
print('Setup complete. Models will be saved to:', os.path.abspath(MODEL_DIR))

In [ ]:
data = fetch_openml('diabetes', version=1, as_frame=True, parser='auto')
df = data.data.copy()
print('Feature columns:', df.columns.tolist())
print('Target name:', data.target.name)
print('Target unique values:', data.target.unique())
print('Shape:', df.shape)
df.head()

In [ ]:
# Binarize target
df['target'] = (data.target.astype(str).str.strip() == 'tested_positive').astype(int)
print('Class distribution:')
print(df['target'].value_counts())

# Replace physiologically impossible zeros with median
impute_cols = ['plas', 'pres', 'skin', 'insu', 'mass']
for col in impute_cols:
    if col in df.columns:
        df[col] = df[col].replace(0, np.nan)
df = df.fillna(df.median(numeric_only=True))

feature_names = [c for c in df.columns if c != 'target']
X = df[feature_names].astype(float).values
y = df['target'].values
print('Features:', feature_names)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
print('Training complete')

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Non-diabetic', 'Diabetic']))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc')
print(f'5-Fold CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
plt.title('SHAP Feature Importance - Diabetes')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/diabetes_shap.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP plot saved')

In [ ]:
joblib.dump(model,         f'{MODEL_DIR}/diabetes_model.pkl')
joblib.dump(scaler,        f'{MODEL_DIR}/diabetes_scaler.pkl')
joblib.dump(feature_names, f'{MODEL_DIR}/diabetes_features.pkl')
for fname in ['diabetes_model.pkl', 'diabetes_scaler.pkl', 'diabetes_features.pkl']:
    size = os.path.getsize(f'{MODEL_DIR}/{fname}') / 1024
    print(f'  {fname}  ({size:.1f} KB)')
print('All diabetes artifacts saved!')